# Media Framing of Palestine – Progress Report

**Course:** CS 418 - Introduction to Data Science  
**Group:** 404 Data Not Found  
**Group Members:** Bryan Dominguez, Sami Fariz, Safe Nassan, Justin McClain, Baraa Eldeirawi

This project analyzes how international news outlets frame the Palestine-Israel conflict using sentiment analysis and machine learning techniques.

# Project Introduction
## Research Question
How does the tone of international media coverage compare and differ across regions, and what does this showcase about media framing and potential bias?

## Dataset
To investigate our question, we use two primary sources: GDELT and NewsAPI. GDELT provides large-scale global coverage across multiple countries and time periods. NewsAPI allows us to gather more recent articles from established news outlets.

Each article in the dataset includes key attributes such as the title, text content, source, and publication date. We then filter the articles based on specific keywords such as “Palestine,” “Gaza,” “West Bank,” and “Israel conflict” to make sure they are relevant to the research topic.

## Motivation
Understanding how media framing differs across regions is important because news coverage influences public opinion, political perspectives, and policy discussions. If different outlets present the same things using different tones or language, it leads to varying interpretations.

By applying data analysis, sentiment analysis, and machine learning techniques, this project seeks to provide a data-driven perspective on how narratives surrounding the Palestine-Israel conflict may differ across media sources.


# Changes from Initial Proposal
The overall goal remains the same: to analyze how media coverage of Palestine differs across media sources and regions. The only thing that changed was refining the data we were able to collect so that we could implement the requirements for the Progress Report.

One important adjustment is that our analysis is currently more focused on sentiment and textual patterns than on broader claims about media bias. Rather than trying to fully explain bias at this stage, we are using measurable features such as sentiment, word usage, and source differences as indicators of how framing may vary across outlets.

We simplified part of the ML task. We are currently using article text to predict the source for regional patterns. This allows us to establish a baseline workflow and evaluate whether meaningful textual differences exist in the dataset.

Additionally, the data collection process led us to rely on GDELT more heavily for this current milestone, while NewsAPI remains part of the overall project plan for expanding the dataset later. This change was made to keep the progress report concise while still showing the required things in data collection, cleaning, visualization, and machine learning.

## Data Attributes
The articles in the dataset are going to include the following key attributes:
- Title
- Text content or description (TBD)
- Source
- Date of Publication

These attributes let us analyze both the content of articles and the context in which they're published.

## Data Integration
Since data is collected from multiple sources, the datasets are combined into a unified dataset. This makes sure there is consistency in structure and allows us to analyze the articles more easily.
## Scope and Limitations
Since the dataset is large and diverse, we did end up having some limitations:
- NewsAPI primarily includes major English-language outlets, which may introduce bias
- GDELT includes global data, but some articles may be translated, which can affect text quality
- The dataset may not equally represent all regions or perspectives

Despite these limitations, the combined dataset provides a non-biased dataset analyzing differences in media framing across regions.

# Data Cleaning

## Data Structure
The dataset consists of news articles where each row represents a single article. The main columns include the article title, text content, source, and publication date. It is structured this way to support both textual and metadata-based analysis.

## Granularity
The data is collected at the article level, which means each observation corresponds to one news article. This allows for detailed analysis of language and sentiment across different sources.

## Data Cleaning Process
To prepare the dataset for analysis, we took a couple of steps to ensure consistency and quality:

- Duplicate articles were removed to avoid redundancy
- Rows with missing or empty text fields were dropped
- All text was converted to lowercase for consistency
- URLs and special characters were removed
- Stopwords (commonly used words such as “the” and “and”) were removed to focus on meaningful content

These steps help reduce noise in the dataset and improve the performance of both sentiment analysis and machine learning models.

## Handling Missing Values
Missing values in the text field were removed, since text content is needed for analysis. For metadata such as source or date, missing values were either labeled as “unknown” or stayed the same, depending on their impact.

## Data Quality Considerations
While cleaning improves the dataset, we still have some limitations. Some articles may have incomplete descriptions, and differences in formatting may introduce inconsistencies. In addition, translated articles may differ from the original wording, affecting sentiment analysis.

Despite these challenges, the cleaned dataset provides a foundation for us to build upon.

# Exploratory Data Analysis

## Data Overview
Our dataset consists of news articles collected from multiple sources. Each entry includes text content, source information, and publication date. The dataset contains articles from different regions and outlets to provide a diverse perspective on the topic.

## Initial Observations
From initial exploration, several patterns can be observed:
- Some news sources contribute significantly more articles than others, leading to a potential imbalance in the dataset
- The number of articles varies over time, often increasing during major events related to the conflict
- There is variation in article length and structure depending on the source

## Early Insights
Preliminary analysis shows that there could be differences in language and tone across articles from different sources. This strengthens the idea that media framing could vary depending on the region or news outlet.

## Potential Issues
Several challenges were identified during exploration:
- Imbalanced representation of sources and regions
- Missing or incomplete metadata in some articles
- Differences in formatting and text quality across sources

# Hypothesis Testing

## Hypothesis
We hypothesize that news coverage of the Palestine-Israel conflict differs in tone across different sources and can be observed through sentiment analysis of the text.

## Why This is Important
Media framing plays a major role in shaping how audiences interpret global events. If different outlets present more positive or negative sentiment, this could influence the public on who to side with. By analyzing sentiment, we can determine the difference in tone across different articles.

## Approach
To test this hypothesis, we created sentiment scores for each article using NLP techniques. We then create a graph of these sentiment scores to observe whether the dataset leans more positive, negative, or neutral.

## Expected Outcome
We expect to observe variations in sentiment across the dataset, which suggests that news coverage is not similar in tone. This supports the idea that different sources may frame the same events differently.

In [13]:
# Run this once if needed
# %pip install pandas numpy matplotlib scikit-learn nltk textblob langdetect requests scipy

import re
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from IPython.display import display, Markdown

import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.decomposition import LatentDirichletAllocation

from scipy.stats import kruskal

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

ModuleNotFoundError: No module named 'textblob'

In [14]:
KEYWORDS = ["Palestine", "Gaza", "West Bank", "Israel conflict"]
MAX_RECORDS_PER_KEYWORD = 250
RANDOM_STATE = 42

def fetch_gdelt_data(keywords, max_records=250):
    """
    Collect GDELT article metadata.
    We use titles/headlines because they are consistently available across results.
    """
    base_url = "https://api.gdeltproject.org/api/v2/doc/doc"
    rows = []

    for keyword in keywords:
        params = {
            "query": keyword,
            "mode": "ArtList",
            "maxrecords": max_records,
            "format": "json"
        }

        try:
            response = requests.get(base_url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()

            for article in data.get("articles", []):
                rows.append({
                    "keyword": keyword,
                    "title": article.get("title"),
                    "source_country": article.get("sourcecountry"),
                    "date": article.get("seendate"),
                    "url": article.get("url")
                })

        except Exception as e:
            print(f"Error for keyword '{keyword}': {e}")

        time.sleep(1)

    return pd.DataFrame(rows)

def is_english(text):
    if pd.isna(text):
        return False
    try:
        return detect(str(text)) == "en"
    except LangDetectException:
        return False

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

def compute_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return TextBlob(text).sentiment.polarity

def plot_conf_matrix(cm, labels, title):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

def show_top_words_per_class(model, vectorizer, class_names, top_n=10):
    feature_names = np.array(vectorizer.get_feature_names_out())
    for i, cls in enumerate(class_names):
        top_idx = np.argsort(model.coef_[i])[-top_n:][::-1]
        top_words = feature_names[top_idx]
        print(f"{cls}: {', '.join(top_words)}")

def get_top_words(text_series, n=15):
    words = " ".join(text_series).split()
    return Counter(words).most_common(n)

In [15]:
raw_df = fetch_gdelt_data(KEYWORDS, MAX_RECORDS_PER_KEYWORD)

print("Raw shape:", raw_df.shape)
raw_df.head()

Error for keyword 'Palestine': HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Max retries exceeded with url: /api/v2/doc/doc?query=Palestine&mode=ArtList&maxrecords=250&format=json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x14d4592b0>, 'Connection to api.gdeltproject.org timed out. (connect timeout=30)'))
Error for keyword 'Gaza': HTTPSConnectionPool(host='api.gdeltproject.org', port=443): Max retries exceeded with url: /api/v2/doc/doc?query=Gaza&mode=ArtList&maxrecords=250&format=json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x14d433390>, 'Connection to api.gdeltproject.org timed out. (connect timeout=30)'))
Raw shape: (500, 5)


,keyword,title,source_country,date,url
0,West Bank,"BM , işgal altındaki Batı Şerianın İsrail tara...",Turkey,20260211T084500Z,https://www.haberler.com/guncel/bm-den-israil-...
1,West Bank,Δυτική Όχθη : Ο ΟΗΕ καταδικάζει την επέκταση τ...,Greece,20260211T173000Z,https://www.newsit.gr/kosmos/dytiki-oxthi-o-oi...
2,West Bank,СМИ : ЦАХАЛ перебрасывает войска с севера на З...,Russia,20260323T220000Z,https://ria.ru/20260324/voyska-2082517510.html
3,West Bank,Batı Şerianın ilhakını desteklemiyorum,Turkey,20260210T230000Z,https://www.haberturk.com/bati-seria-nin-ilhak...
4,West Bank,"Trump , İsrailin Batı Şeriayı ilhak etmeye yön...",Turkey,20260210T231500Z,http://www.haberler.com/guncel/trump-bati-seri...


In [17]:
# -----------------------------
# Data cleaning and preparation
# -----------------------------

raw_rows = len(raw_df)

df = raw_df.copy()

# Drop exact duplicates based on title, country, and date
df = df.drop_duplicates(subset=["title", "source_country", "date"])
after_dedup = len(df)

# Drop missing core fields
df = df.dropna(subset=["title", "source_country", "date"]).copy()
after_missing = len(df)

# Parse dates
df["date"] = pd.to_datetime(df["date"], format="%Y%m%dT%H%M%SZ", errors="coerce")
df = df.dropna(subset=["date"]).copy()
after_dates = len(df)

# Keep English-language titles only
df["is_english"] = df["title"].apply(is_english)
df = df[df["is_english"]].copy()
after_english = len(df)

# Clean text
df["clean_text"] = df["title"].apply(clean_text)
df["headline_length_words"] = df["clean_text"].str.split().apply(len)

# Keep rows with enough text to analyze
df = df[df["headline_length_words"] >= 3].copy()
after_min_words = len(df)

# Use the top 3 source countries by count
country_counts = df["source_country"].value_counts()
top_countries = country_counts.head(3).index.tolist()

df_selected = df[df["source_country"].isin(top_countries)].copy()

# Balance the selected dataset for ML fairness
selected_counts = df_selected["source_country"].value_counts()
min_count = selected_counts.min()

df_balanced = (
    df_selected.groupby("source_country", group_keys=False)
    .apply(lambda x: x.sample(n=min_count, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

df_selected["month"] = df_selected["date"].dt.to_period("M").astype(str)
df_balanced["month"] = df_balanced["date"].dt.to_period("M").astype(str)

final_rows_selected = len(df_selected)
final_rows_balanced = len(df_balanced)

# Save cleaned datasets for the final project submission
df_selected.to_csv("cleaned_palestine_headlines_selected.csv", index=False)
df_balanced.to_csv("cleaned_palestine_headlines_balanced.csv", index=False)

print("Top countries used:", top_countries)
print("Selected dataset shape:", df_selected.shape)
print("Balanced dataset shape:", df_balanced.shape)

df_balanced.head()

NameError: name 'LangDetectException' is not defined

In [18]:
date_min = df_selected["date"].min().date()
date_max = df_selected["date"].max().date()

display(Markdown(f"""
## Data Collection and Cleaning Summary

**Data source used:** GDELT Document API  
**Keywords used:** {", ".join(KEYWORDS)}  
**Unit of analysis:** one English-language headline/article title  
**Final label used for analysis:** `source_country`

### Before/After Cleaning Counts
- Raw collected rows: **{raw_rows}**
- After removing duplicates: **{after_dedup}**
- After dropping missing title/country/date: **{after_missing}**
- After valid date parsing: **{after_dates}**
- After English-only filtering: **{after_english}**
- After minimum word-length filter: **{after_min_words}**

### Final Analysis Scope
- Top source countries selected: **{", ".join(top_countries)}**
- Date range in selected dataset: **{date_min}** to **{date_max}**
- Rows in selected dataset: **{final_rows_selected}**
- Rows per country in balanced ML dataset: **{min_count}**
- Rows in balanced dataset: **{final_rows_balanced}**

### Why the data was cleaned this way
For the final project, we narrowed the dataset to **English-language headlines** so that sentiment analysis would be more reliable and the text would be more consistent across countries. We also simplified the machine learning task by using only the **top three source countries** and balancing the dataset before training the classifiers.
"""))

NameError: name 'df_selected' is not defined

In [19]:
summary_table = pd.DataFrame({
    "Metric": [
        "Rows in selected dataset",
        "Rows in balanced dataset",
        "Unique source countries in selected dataset",
        "Date range start",
        "Date range end",
        "Average cleaned headline length"
    ],
    "Value": [
        final_rows_selected,
        final_rows_balanced,
        df_selected["source_country"].nunique(),
        str(date_min),
        str(date_max),
        round(df_selected["headline_length_words"].mean(), 2)
    ]
})

country_table = df_selected["source_country"].value_counts().reset_index()
country_table.columns = ["source_country", "article_count"]

length_table = (
    df_selected.groupby("source_country")["headline_length_words"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
    .reset_index()
)

month_table = (
    df_selected.groupby("month")
    .size()
    .reset_index(name="article_count")
    .sort_values("month")
)

display(summary_table)
display(country_table)
display(length_table)
display(month_table.head(12))

NameError: name 'final_rows_selected' is not defined

# Machine Learning Analysis

## Objective
The goal for the use of ML is to determine whether the text of the news articles has enough information to predict the source. We use the source as a proxy to determine regional differences in media coverage.

# Usage
We convert the cleaned text into a number using TFIDF. This makes sure to capture the importance of words in the article. We then used a regression model to classify articles based on their sources. This makes it so that the model can detect patterns in the text, when looking at different sources.

# Importance
This matters because if the model is able to predict the source using only the text, this means that different news outlets use distinct language patterns, which supports the idea that media framing varies across sources.


In [ ]:
# Visualization 1: article count by source country

count_data = df_selected["source_country"].value_counts().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(count_data.index, count_data.values)
plt.title("Number of Palestine-Related Headlines by Source Country")
plt.xlabel("Source Country")
plt.ylabel("Number of Headlines")
plt.tight_layout()
plt.show()

top_country = count_data.idxmax()
top_country_count = int(count_data.max())
bottom_country = count_data.idxmin()
bottom_country_count = int(count_data.min())

display(Markdown(f"""
### Visualization 1: Coverage Volume by Country

**Hypothesis:** Coverage volume differs across source countries, which may shape which viewpoints dominate the dataset.

**Takeaway:** In the selected dataset, **{top_country}** contributes the most headlines (**{top_country_count}**), while **{bottom_country}** contributes the fewest (**{bottom_country_count}**). This shows that the data is not naturally balanced across countries, which is why a balanced version of the dataset is used later for machine learning.
"""))

In [ ]:
# Sentiment analysis on the balanced dataset for fairer comparison

df_balanced["sentiment"] = df_balanced["clean_text"].apply(compute_sentiment)

sentiment_means = (
    df_balanced.groupby("source_country")["sentiment"]
    .mean()
    .sort_values()
)

boxplot_data = [
    df_balanced.loc[df_balanced["source_country"] == c, "sentiment"]
    for c in sentiment_means.index
]

plt.figure(figsize=(9, 5))
plt.boxplot(boxplot_data, labels=sentiment_means.index)
plt.title("Sentiment Distribution by Source Country")
plt.xlabel("Source Country")
plt.ylabel("Sentiment Polarity")
plt.tight_layout()
plt.show()

highest_sent_country = sentiment_means.idxmax()
lowest_sent_country = sentiment_means.idxmin()
highest_sent_value = float(sentiment_means.max())
lowest_sent_value = float(sentiment_means.min())

display(Markdown(f"""
### Visualization 2: Sentiment by Country

**Hypothesis:** The tone of Palestine-related headlines differs across source countries.

**Takeaway:** In the balanced dataset, **{highest_sent_country}** has the highest average sentiment score (**{highest_sent_value:.3f}**), while **{lowest_sent_country}** has the lowest (**{lowest_sent_value:.3f}**). This suggests that tone varies across countries, even after balancing the number of headlines per group.
"""))

In [ ]:
# Statistical analysis: test whether sentiment differs across countries

groups = [
    df_balanced.loc[df_balanced["source_country"] == c, "sentiment"].values
    for c in sentiment_means.index
]

kruskal_stat, kruskal_p = kruskal(*groups)

display(Markdown(f"""
## Statistical Analysis: Kruskal-Wallis Test

We used a **Kruskal-Wallis test** to compare sentiment across the selected source countries.

- Test statistic: **{kruskal_stat:.4f}**
- p-value: **{kruskal_p:.4f}**

Interpretation: {'The p-value is below 0.05, which suggests that at least one country has a statistically different sentiment distribution.' if kruskal_p < 0.05 else 'The p-value is not below 0.05, so we do not have strong evidence that sentiment distributions differ statistically across the selected countries in this sample.'}
"""))

In [ ]:
# Extra visualization: article counts over time by source country

monthly_counts = (
    df_selected.groupby(["month", "source_country"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

monthly_counts.plot(marker="o", figsize=(10, 5))
plt.title("Monthly Number of Palestine-Related Headlines by Source Country")
plt.xlabel("Month")
plt.ylabel("Headline Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(Markdown("""
### Extra Visualization: Coverage Over Time

This line chart shows how the number of Palestine-related headlines changes over time for each selected source country. It helps show whether spikes in coverage happen at the same time across countries or whether some countries respond more strongly during certain periods.
"""))

In [ ]:
# Prepare train/test split

X = df_balanced["clean_text"]
y = df_balanced["source_country"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=1)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Baseline
baseline_label = y_train.mode()[0]
baseline_preds = np.full(shape=len(y_test), fill_value=baseline_label)
baseline_acc = accuracy_score(y_test, baseline_preds)

# Logistic Regression
log_reg = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
log_reg.fit(X_train_tfidf, y_train)
log_preds = log_reg.predict(X_test_tfidf)
log_acc = accuracy_score(y_test, log_preds)

print("Baseline label:", baseline_label)
print("Baseline accuracy:", round(baseline_acc, 4))
print("Logistic Regression accuracy:", round(log_acc, 4))
print()
print("Logistic Regression classification report:")
print(classification_report(y_test, log_preds, zero_division=0))

In [ ]:
log_cm = confusion_matrix(y_test, log_preds, labels=log_reg.classes_)
plot_conf_matrix(log_cm, list(log_reg.classes_), "Logistic Regression Confusion Matrix")

display(Markdown(f"""
### Logistic Regression Interpretation

The logistic regression model achieved an accuracy of **{log_acc:.3f}**, while the baseline accuracy was **{baseline_acc:.3f}**.

{'Because the logistic regression model performed better than the baseline, this suggests that the headline text contains country-specific language patterns that help distinguish the selected source countries.' if log_acc > baseline_acc else 'Because the logistic regression model did not outperform the baseline, this suggests that the current text features are still limited or that the selected headlines are not distinct enough for strong classification.'}

This result matters because it gives a measurable way to test whether framing differences appear in the text itself rather than only in manual interpretation.
"""))

In [ ]:
# Top predictive words for each class in Logistic Regression
print("Top predictive words for each source country:\n")
show_top_words_per_class(log_reg, vectorizer, log_reg.classes_, top_n=10)

In [ ]:
# Multinomial Naive Bayes

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

print("Naive Bayes accuracy:", round(nb_acc, 4))
print()
print("Naive Bayes classification report:")
print(classification_report(y_test, nb_preds, zero_division=0))

In [ ]:
nb_cm = confusion_matrix(y_test, nb_preds, labels=nb_model.classes_)
plot_conf_matrix(nb_cm, list(nb_model.classes_), "Naive Bayes Confusion Matrix")

comparison_df = pd.DataFrame({
    "Model": ["Baseline", "Logistic Regression", "Naive Bayes"],
    "Accuracy": [baseline_acc, log_acc, nb_acc]
}).sort_values("Accuracy", ascending=False)

display(comparison_df)

best_model = comparison_df.iloc[0]["Model"]
best_acc = comparison_df.iloc[0]["Accuracy"]

display(Markdown(f"""
### Model Comparison

The three models were compared using the same train/test split. The best-performing approach in this notebook is **{best_model}** with an accuracy of **{best_acc:.3f}**.

Comparing both ML models to the baseline is important because it shows whether the model is actually learning from the text or just benefiting from class distribution.
"""))

In [ ]:
# Extra deliverable: LDA topic modeling

count_vectorizer = CountVectorizer(max_features=1500, stop_words="english", min_df=1)
X_counts = count_vectorizer.fit_transform(df_balanced["clean_text"])

n_topics = 3
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=RANDOM_STATE,
    learning_method="batch"
)
doc_topic = lda.fit_transform(X_counts)

feature_names = count_vectorizer.get_feature_names_out()

topic_rows = []
for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_terms = [feature_names[i] for i in top_indices]
    topic_rows.append({
        "Topic": f"Topic {topic_idx + 1}",
        "Top Words": ", ".join(top_terms)
    })

topic_keywords_df = pd.DataFrame(topic_rows)
display(topic_keywords_df)

In [ ]:
df_topics = df_balanced.copy()
df_topics["dominant_topic"] = doc_topic.argmax(axis=1)

topic_country = pd.crosstab(
    df_topics["source_country"],
    df_topics["dominant_topic"],
    normalize="index"
)

topic_country.columns = [f"Topic {i+1}" for i in topic_country.columns]

topic_country.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Dominant Topic Distribution by Source Country")
plt.xlabel("Source Country")
plt.ylabel("Proportion of Headlines")
plt.legend(title="Topic")
plt.tight_layout()
plt.show()

display(Markdown("""
## Extra Deliverable: Topic Modeling

LDA topic modeling was added as an extra deliverable to identify recurring themes in the headlines. The topic distribution plot helps show whether some countries emphasize different groups of words or themes more than others.
"""))

In [ ]:
print("Saved files:")
print("- cleaned_palestine_headlines_selected.csv")
print("- cleaned_palestine_headlines_balanced.csv")

# Model Interpretation


The logistic regression model achieved an accuracy of 0.25, while the baseline accuracy was 0.50. This indicates that the model is not performing well and is unable to learn meaningful patterns from the data to predict the source of the articles.

One reason for this is that the dataset contains limited textual information, as it primarily uses article titles and timestamps rather than full article content. Additionally, the "source" field represents country information rather than specific news outlets, making the classification task more difficult.

These results suggest that while machine learning has potential for analyzing media framing, improvements in data quality and feature selection are needed for better performance.

# Reflection

The most difficult part of the project overall has been collecting data and turning it into a clean, consistent dataset for analysis. Because we are collecting articles from multiple sources, the source names, text fields, and labels do not always match, which is why preparation took longer than expected. 

Our idea that media framing can vary depending on the article's source is supported by one of our initial findings which is that there seem to be differences in tone and language between sources. Additionally, since we were able to clean the data, perform sentiment analysis, create a visualization, and test a machine learning model with a baseline for comparison, we now have some measurable results. 

However, one of our main issues at the moment is that the dataset is still tiny and uneven, which makes it more difficult to get good model performance. I think we are mostly on track because we already have a working pipeline and some initial findings. We still have to give more time to improving the dataset and improving the analysis, though. I do think the effort is justified based on what we have seen so far because the topic is important, the data shows some real patterns, and the results should become much more reliable with more data and clearer labels.


# Next Step

We will next collect more articles and improve the dataset to make the content and source information more accurate and consistent. We also hope to provide more accurate comparisons between sources by investigating differences in tone and wording.  We want to enhance machine learning results and increase the accuracy of the visualizations by using a larger and more balanced dataset.  We will know we are making progress if our model performs better than the baseline and our graphs show clearer patterns.